# Fake News Detector

In [1]:
# Basic imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
# Pre-processing
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import hstack
# Model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aparajaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Data Loading 
* Load and examine dataset
* Convert datatypes
* Treat missing values

In [2]:
df = pd.read_csv("Dataset.csv")

In [3]:
# Examining Dataset 
print("Dataset shape:",df.shape)
print("\nSample: ")
df.head(1)

Dataset shape: (72134, 4)

Sample: 


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1


In [4]:
# Checking and converting datatypes
print("Original Datatypes:\n",df.dtypes)
df['title'] = df['title'].astype('string')
df['text'] = df['text'].astype('string')
print("\nChanged Datatypes:\n",df.dtypes)

Original Datatypes:
 Unnamed: 0     int64
title         object
text          object
label          int64
dtype: object

Changed Datatypes:
 Unnamed: 0             int64
title         string[python]
text          string[python]
label                  int64
dtype: object


In [5]:
# Treating missing values
print("Number of missing values per column:\n",df.isna().sum())
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
print("\nAfter treating missing values-")
print("Number of missing values per column:\n",df.isna().sum())

Number of missing values per column:
 Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

After treating missing values-
Number of missing values per column:
 Unnamed: 0    0
title         0
text          0
label         0
dtype: int64


## Preprocessing

In [6]:
def preprocess(df, remove_stopwords=True, remove_sourcewords=False, remove_punctuation=True, lowercase=True, check= False):
    df = df.copy()
    # Convert to Lowercase
    if lowercase:
        df['text'] = df['text'].str.lower()
        df['title'] = df['title'].str.lower()

    # Remove punctuation
    if remove_punctuation: 
        df['title'] = df['title'].str.replace(r'[^\w\s]+', '', regex=True) 
        df['text'] = df['text'].str.replace(r'[^\w\s]+', '', regex=True) 

    # Remove stopwords 
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))

        df['text'] = df['text'].apply(
            lambda x: " ".join([word for word in x.split() if word not in stop_words])
        )
        df['title'] = df['title'].apply(
            lambda x: " ".join([word for word in x.split() if word not in stop_words])
        )

    # Remove article source names
    if remove_sourcewords:
        source_phrases = [
            "washington reuters",
            "new york times",
            "associated press",
            "wall street journal",
            "fox news",
            "breitbart",
            "cnn",
            "bbc",
            "reuters"
        ]

        pattern = r'\b(?:' + '|'.join(map(re.escape, source_phrases)) + r')\b'

        df['text'] = df['text'].str.replace(pattern, '', regex=True)
        df['title'] = df['title'].str.replace(pattern, '', regex=True)

        df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
        df['title'] = df['title'].str.replace(r'\s+', ' ', regex=True).str.strip()

    # Check if pre-processing caused too many empty rows
    if check: 
        print("No of empty title rows: ",(df['title'].str.len() == 0).sum())
        print("No of empty text rows: ",(df['text'].str.len() == 0).sum())
        count = ((df['title'].str.len() == 0) & (df['text'].str.len() == 0)).sum()
        print("No of rows with empty title AND text: ",count)
    
    return df

## Vectorization 

In [7]:
def vectorize(df, method, mdf=5, ngramRange=(1,2), max_features_title=50000, max_features_text=500000, vec_text = True, vec_title = True, verbose = False):
    X = df[["title", "text"]]
    Y = df["label"]

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.33, random_state=42)

    # Initialize the vectorizer
    if method == "tfidf":
        vectorizer_title = TfidfVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_title)
        vectorizer_text = TfidfVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_text)
    elif method == "count":
        vectorizer_title = CountVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_title)
        vectorizer_text = CountVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_text)
    
    # Vectorize text and/or title
    if vec_text and vec_title:
        # Fit and transform the data
        matrix_title_train = vectorizer_title.fit_transform(X_train["title"])
        matrix_title_test = vectorizer_title.transform(X_test["title"])

        matrix_text_train = vectorizer_text.fit_transform(X_train["text"])
        matrix_text_test = vectorizer_text.transform(X_test["text"])

        X_train = hstack([matrix_title_train, matrix_text_train])
        X_test = hstack([matrix_title_test, matrix_text_test])

        feature_names = np.concatenate((
        vectorizer_title.get_feature_names_out(),
        vectorizer_text.get_feature_names_out()
        ))
    elif vec_title:
        # Fit and transform the data
        matrix_title_train = vectorizer_title.fit_transform(X_train["title"])
        matrix_title_test = vectorizer_title.transform(X_test["title"])
        
        X_train = matrix_title_train
        X_test = matrix_title_test

        feature_names = vectorizer_title.get_feature_names_out()
    elif vec_text:
        # Fit and transform the data
        matrix_text_train = vectorizer_text.fit_transform(X_train["text"])
        matrix_text_test = vectorizer_text.transform(X_test["text"])

        X_train = matrix_text_train
        X_test = matrix_text_test

        feature_names = vectorizer_text.get_feature_names_out()


    # Matrix inspection
    # Shape = (number_of_documents, number_of_features)
    if verbose:
        print(X_train.shape)
        print(X_test.shape)

    total_features = X_train.shape[1]
    return X_train, X_test, Y_train, Y_test, feature_names, total_features

## Evaluation

In [8]:
def evaluate(Y_test, Y_pred, model, feature_names, plot_cm= False, verbose=True):
    # Metrics
    accuracy = accuracy_score(Y_test, Y_pred)
    precision = precision_score(Y_test, Y_pred)
    recall = recall_score(Y_test, Y_pred)
    f1 = f1_score(Y_test, Y_pred, average="macro")

    # Confusion Matrix
    cm = confusion_matrix(Y_test, Y_pred)

    # Top positive and negative features
    coefficients = model.coef_[0] 

    df_features = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefficients
    })

    top_negative_features = df_features.sort_values(by='Coefficient', ascending=True).head(5)
    top_positive_features = df_features.sort_values(by='Coefficient', ascending=False).head(5)

    # Display evaluation results
    if verbose:
        print("Accuracy: ", accuracy)
        print("Precision: ", precision)
        print("Recall: ", recall)
        print("F1: ", f1)

        print("\nConfusion Matrix:\n", cm)

        if plot_cm:
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
            disp.plot(cmap=plt.cm.Blues)
            plt.show()

        print("\nTop 5 Negative Features:")
        print(top_negative_features)

        print("\nTop 5 Positive Features:")
        print(top_positive_features)

    return accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features

# Experimenting parameters

In [9]:
def run_experiment(df, vectorizer, experiment_name="", ngramRange=(1,2), remove_stopwords=True, vec_title=True, vec_text=True, mdf=5, remove_sourcewords=False):
    # Creating features
    start_time = time.perf_counter()

    df = preprocess(df,remove_stopwords=remove_stopwords, remove_sourcewords=remove_sourcewords)
    X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df, method = vectorizer, ngramRange=ngramRange, vec_title=vec_title, vec_text=vec_text, mdf=mdf)

    end_time = time.perf_counter()
    creation_time = end_time - start_time

    print(f"\nFeatures created in {creation_time:.4f} seconds.")
    print("Total features = ",total_features)

    # Training model
    start_time = time.perf_counter()

    model = LogisticRegression(random_state=42).fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    end_time = time.perf_counter()
    training_time = end_time - start_time

    print(f"Training completed in {training_time:.4f} seconds.")
    
    # Evaluation
    accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = evaluate(Y_test, Y_pred, model, feature_names)

    return {
        "Experiment": experiment_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Features": total_features,
        "Feature Time (s)": creation_time,
        "Training Time (s)": training_time
    }

## Models

## Model 1 (Baseline)
Model - Logistic Regression     

Vectorization - TF-IDF with N-grams

Features -      
* TF-IDF 
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time : 43.2s   
* Training Time : 2.5s                

Metrics -           
* Accuracy:  0.96782
* Precision:  0.95957
* Recall:  0.97820
* F1:  0.96779
* Confusion Matrix:         
 [11148     501        -> False Positives            
   265      11891]     -> False Negatives              

In [10]:
# Creating features
start_time = time.perf_counter()

df = preprocess(df, check=True)
X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df, method = "tfidf")

end_time = time.perf_counter()
creation_time = end_time - start_time

print(f"\nFeatures created in {creation_time:.4f} seconds.")
print("Total features = ",total_features)

No of empty title rows:  563
No of empty text rows:  786
No of rows with empty title AND text:  1

Features created in 42.3964 seconds.
Total features =  477076


In [11]:
# Training model
start_time = time.perf_counter()

model = LogisticRegression(random_state=42).fit(X_train, Y_train)
Y_pred = model.predict(X_test)

end_time = time.perf_counter()
training_time = end_time - start_time

print(f"Training completed in {training_time:.4f} seconds.")

Training completed in 2.5473 seconds.


In [12]:
# Evaluation
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = evaluate(Y_test, Y_pred, model, feature_names)

Accuracy:  0.9678218861583701
Precision:  0.9595706907682375
Recall:  0.9782000658111221
F1:  0.9677905081677439

Confusion Matrix:
 [[11148   501]
 [  265 11891]]

Top 5 Negative Features:
                   Feature  Coefficient
352973             reuters   -19.644100
361547                said   -18.528369
1810             breitbart   -14.876882
432972              trumps    -9.132123
454520  washington reuters    -7.019568

Top 5 Positive Features:
         Feature  Coefficient
15125      video    10.903366
446457       via     8.507856
1777    breaking     6.366823
205873     image     6.172324
197585   hillary     5.396140


In [13]:
# Storing results
results = []

results.append({
    "Experiment": "Baseline",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Features": total_features,
    "Feature Time (s)": creation_time,
    "Training Time (s)": training_time
})

## Experiments
* Unigram vs Unigram+Bigram
* Stopwords ON vs OFF
* Feature Engineering
    - Title only
    - Text only
    - Title and Text combined (post and pre-vectorization)
* Varying min df
    - 2
    - 5
    - 10
* Removing obvious news source identifiers

### Experiment 1 (Unigram vs Unigram+Bigram)

In [14]:
results.append(
    run_experiment(
        df,
        vectorizer="tfidf",
        experiment_name="Unigrams Only",
        ngramRange=(1,1)
    )
)


Features created in 11.6061 seconds.
Total features =  70042
Training completed in 0.5398 seconds.
Accuracy:  0.9671917664356228
Precision:  0.9603399433427762
Recall:  0.976061204343534
F1:  0.9671628837188619

Confusion Matrix:
 [[11159   490]
 [  291 11865]]

Top 5 Negative Features:
         Feature  Coefficient
54379    reuters   -22.127018
55638       said   -16.541791
1202   breitbart   -14.349670
9395        york    -9.069315
63968     trumps    -8.879073

Top 5 Positive Features:
        Feature  Coefficient
9014      video    10.279871
66079       via     9.882855
36281     image     6.699478
1196   breaking     6.195092
46844   october     5.642485


### Experiment 2 (Stopwords ON vs OFF)

In [15]:
results.append(
    run_experiment(
        df, 
        vectorizer="tfidf",
        remove_stopwords=False,
        experiment_name="Kept Stopwords"
    )
)


Features created in 37.1072 seconds.
Total features =  477076
Training completed in 2.5228 seconds.
Accuracy:  0.9678218861583701
Precision:  0.9595706907682375
Recall:  0.9782000658111221
F1:  0.9677905081677439

Confusion Matrix:
 [[11148   501]
 [  265 11891]]

Top 5 Negative Features:
                   Feature  Coefficient
352973             reuters   -19.644100
361547                said   -18.528369
1810             breitbart   -14.876882
432972              trumps    -9.132123
454520  washington reuters    -7.019568

Top 5 Positive Features:
         Feature  Coefficient
15125      video    10.903366
446457       via     8.507856
1777    breaking     6.366823
205873     image     6.172324
197585   hillary     5.396140


### Experiment 3 (Title Only)

In [16]:
results.append(
    run_experiment(
        df,
        vectorizer="tfidf",
        experiment_name="Title Only",
        vec_text=False
    )
)


Features created in 5.0473 seconds.
Total features =  15985
Training completed in 0.0441 seconds.
Accuracy:  0.8973745011552194
Precision:  0.8860799745607759
Recall:  0.9169134583744653
F1:  0.8972176281700783

Confusion Matrix:
 [[10216  1433]
 [ 1010 11146]]

Top 5 Negative Features:
          Feature  Coefficient
1810    breitbart   -17.018015
15915  york times    -8.657581
11862        says    -7.381339
4836      factbox    -5.977784
15911        york    -5.789854

Top 5 Positive Features:
        Feature  Coefficient
15125     video    15.889239
1777   breaking     7.742029
6293    hillary     6.512331
15407     watch     6.439370
14607    tweets     4.665083


### Experiment 4 (Text Only)

In [17]:
results.append(
    run_experiment(
        df,
        vectorizer="tfidf",
        experiment_name="Text Only",
        vec_title=False
    )
)


Features created in 39.3243 seconds.
Total features =  461091
Training completed in 1.1384 seconds.
Accuracy:  0.9554295316110061
Precision:  0.9477762531277747
Recall:  0.9659427443237907
F1:  0.9553864200109012

Confusion Matrix:
 [[11002   647]
 [  414 11742]]

Top 5 Negative Features:
                   Feature  Coefficient
336988             reuters   -24.167961
345562                said   -20.436112
416987              trumps   -10.430314
438535  washington reuters    -8.664267
420072             twitter    -8.057454

Top 5 Positive Features:
          Feature  Coefficient
430472        via    11.603038
181600    hillary     7.371793
189888      image     7.277764
189968  image via     6.267579
269976      obama     5.814341


### Experiment 5 (Text + Title combined before vectorization)

In [18]:
df_combined = pd.DataFrame({
    'title': df['title'],
    'text': df['title'].fillna('') + " " + df['text'].fillna(''),
    'label': df['label'] 
})

results.append(
    run_experiment(
        df_combined,
        vectorizer="tfidf",
        experiment_name="Text and Title combined",
        vec_title=False
    )
)


Features created in 41.0475 seconds.
Total features =  471771
Training completed in 0.6922 seconds.
Accuracy:  0.9579920184835119
Precision:  0.9524659312134978
Recall:  0.9659427443237907
F1:  0.9579578135720492

Confusion Matrix:
 [[11063   586]
 [  414 11742]]

Top 5 Negative Features:
                   Feature  Coefficient
344318             reuters   -23.280788
353116                said   -19.821144
55807            breitbart   -11.314811
426380              trumps    -9.295632
448812  washington reuters    -8.894122

Top 5 Positive Features:
          Feature  Coefficient
440262        via    11.479507
440914      video    10.446018
185594    hillary     8.053941
194136      image     7.538882
194219  image via     6.420594


### Experiment 6 (min_df=2)

In [19]:
results.append(
    run_experiment(
        df,
        vectorizer="tfidf",
        experiment_name="min_df=2",
        mdf=2
    )
)


Features created in 40.3751 seconds.
Total features =  550000
Training completed in 1.0615 seconds.
Accuracy:  0.9675278302877547
Precision:  0.959919191919192
Recall:  0.9772128989799276
F1:  0.9674975993019134

Confusion Matrix:
 [[11153   496]
 [  277 11879]]

Top 5 Negative Features:
           Feature  Coefficient
412950     reuters   -19.861736
422179        said   -18.495374
3627     breitbart   -16.025967
500749      trumps    -9.109845
49681   york times    -7.139926

Top 5 Positive Features:
         Feature  Coefficient
45708      video    11.071569
515119       via     8.534276
3558    breaking     6.268860
255233     image     6.075297
246305   hillary     5.324567


### Experiment 7 (min_df=10)

In [20]:
results.append(
    run_experiment(
        df,
        vectorizer="tfidf",
        experiment_name="min_df=10",
        mdf=10
    )
)


Features created in 39.6384 seconds.
Total features =  195987
Training completed in 0.8453 seconds.
Accuracy:  0.9684940138626339
Precision:  0.9608112475759535
Recall:  0.9782000658111221
F1:  0.9684646014415952

Confusion Matrix:
 [[11164   485]
 [  265 11891]]

Top 5 Negative Features:
           Feature  Coefficient
143912     reuters   -18.955445
147508        said   -17.695253
899      breitbart   -14.498790
177675      trumps    -9.349118
7832    york times    -6.938440

Top 5 Positive Features:
         Feature  Coefficient
7453       video    10.383834
183475       via     8.543894
886     breaking     6.693966
84237      image     6.145000
80978    hillary     5.431574


### Experiment 8 (Removing obvious source identifiers like reuters,breitbart, washington reuters)

In [21]:
results.append(
    run_experiment(
        df,
        vectorizer="tfidf",
        experiment_name="Removed source names",
        remove_sourcewords=True
    )
)


Features created in 46.2760 seconds.
Total features =  475888
Training completed in 1.0045 seconds.
Accuracy:  0.95765595463138
Precision:  0.9547976501305483
Recall:  0.9626521882198091
F1:  0.9576284049980466

Confusion Matrix:
 [[11095   554]
 [  454 11702]]

Top 5 Negative Features:
                 Feature  Coefficient
360372              said   -19.927960
431789            trumps    -9.823138
434872           twitter    -7.824752
167629            follow    -7.080782
321424  president donald    -6.758858

Top 5 Positive Features:
         Feature  Coefficient
14943      video    12.045298
445271       via     9.536406
1755    breaking     7.175800
205385     image     6.990806
197102   hillary     6.085774


In [22]:
comparison_df = pd.DataFrame(results)
comparison_df.sort_values("Accuracy", ascending=False)

,Experiment,Accuracy,Precision,Recall,F1,Features,Feature Time (s),Training Time (s)
7,min_df=10,0.968494,0.960811,0.978200,0.968465,195987,39.638384,0.845261
0,Baseline,0.967822,0.959571,0.978200,0.967791,477076,42.396362,2.547286
2,Kept Stopwords,0.967822,0.959571,0.978200,0.967791,477076,37.107182,2.522789
6,min_df=2,0.967528,0.959919,0.977213,0.967498,550000,40.375067,1.061482
1,Unigrams Only,0.967192,0.960340,0.976061,0.967163,70042,11.606108,0.539817
5,Text and Title combined,0.957992,0.952466,0.965943,0.957958,471771,41.047531,0.692221
8,Removed source names,0.957656,0.954798,0.962652,0.957628,475888,46.275968,1.004536
4,Text Only,0.955430,0.947776,0.965943,0.955386,461091,39.324295,1.138381
3,Title Only,0.897375,0.886080,0.916913,0.897218,15985,5.047273,0.044095


## Model Analysis and Experimental Findings

### Baseline Performance

The baseline model uses Logistic Regression with TF-IDF vectorization, separate feature spaces for titles and article text, stopword removal, unigram and bigram features, and a minimum document frequency (`min_df`) of 5.

**Baseline Metrics**

* Accuracy: 96.78%
* Precision: 95.96%
* Recall: 97.82%
* F1 Score: 96.78%

The model achieved strong and well-balanced performance, with relatively few classification errors. Recall was slightly higher than precision, indicating that the model missed fewer positive samples at the cost of a slightly larger number of false positives.

---

### Feature Importance Analysis

Inspection of the Logistic Regression coefficients revealed several highly influential features.

**Real News Indicators (Negative Coefficients)**

* `reuters`
* `said`
* `washington reuters`
* `new york`
* `york times`
* `president donald`
* `twitter`

**Fake News Indicators (Positive Coefficients)**

* `video`
* `via`
* `breaking`
* `image`
* `image via`
* `hillary`
* `2016`
* `october`
* `trump`
* `november`

Many of the strongest features corresponded to publisher names, news organizations, and source identifiers. This suggested that the model might partially rely on identifying where an article originated rather than exclusively analyzing its content.

At the same time, several highly weighted fake-news indicators were associated with media-sharing, clickbait-style language, or commonly occurring terms within the dataset.

---

### Impact of Source Identifiers

To test whether the model was overly dependent on publisher information, a preprocessing experiment removed several prominent source names, including Reuters, Breitbart, CNN, BBC, Fox News, Associated Press, and New York Times references.

Performance decreased from approximately:

* Accuracy: 96.78% → 95.77%
* F1 Score: 96.78% → 95.76%

The drop was measurable but relatively small.

This suggests that source identifiers contribute useful information, but they are not the primary reason for the model's success. The classifier continues to perform well even when major publisher cues are removed, indicating that it is learning meaningful patterns from the article content itself.

---

### N-Gram Analysis

A comparison between unigram-only features and unigram-plus-bigram features showed only a modest improvement from adding bigrams.

| Configuration      | Accuracy |
| ------------------ | -------- |
| Unigrams Only      | 96.72%   |
| Unigrams + Bigrams | 96.78%   |

While several informative bigrams appeared among the top features (e.g., `washington reuters`, `image via`, `new york`), most influential features remained individual words.

This indicates that the majority of predictive information is contained within unigram features, while bigrams provide additional contextual information and a small performance boost.

However, the increase in feature count was substantial, suggesting a tradeoff between performance and computational efficiency.

---

### Stopword Removal Analysis

An experiment comparing stopword removal against retaining stopwords produced virtually identical results.

This suggests that stopwords contribute little useful information to the classification task. Since TF-IDF naturally assigns low weights to extremely common words, explicit stopword removal appears to have minimal impact on model performance for this dataset.

---

### Title vs Article Content

Several feature-engineering experiments explored the relative importance of article titles and article bodies.

| Configuration                  | Accuracy |
| ------------------------------ | -------- |
| Title Only                     | ~89.7%   |
| Text Only                      | ~95.5%   |
| Title + Text Combined          | ~95.8%   |
| Separate Title + Text Features | ~96.8%   |

The results indicate that article content carries most of the predictive power. Titles alone are surprisingly informative but perform significantly worse than full article text.

Combining titles and text into a single field before vectorization resulted in lower performance than maintaining separate feature spaces.

This suggests that preserving the distinction between titles and article bodies allows the model to extract more useful information from each source.

---

### Effect of Minimum Document Frequency

Different values of `min_df` were tested to evaluate the impact of removing increasingly rare words.

| min_df | Accuracy |
| ------ | -------- |
| 2      | 96.75%   |
| 5      | 96.78%   |
| 10     | 96.85%   |

Interestingly, increasing `min_df` slightly improved performance while significantly reducing the number of features.

This suggests that many rare words contribute noise rather than useful predictive information. Filtering uncommon terms appears to improve generalization while reducing model complexity.

Among the tested values, `min_df=10` produced the strongest results.

---

### Computational Performance

A notable observation throughout experimentation was the difference between feature-generation cost and model-training cost.

Typical timings were:

* TF-IDF Feature Generation: ~40–45 seconds
* Logistic Regression Training: ~0.5–3 seconds

Feature extraction consistently required significantly more computation than model training.

This indicates that the primary bottleneck in the current pipeline is vectorization rather than classification. Future optimization efforts would likely benefit more from reducing feature dimensionality than from changing the classifier itself.

---

### Key Conclusions

* Logistic Regression performs exceptionally well on this dataset using TF-IDF features.
* Source identifiers contribute useful information but are not solely responsible for model performance.
* Article text provides substantially more predictive information than titles alone.
* Keeping title and text as separate feature spaces performs better than merging them before vectorization.
* Stopword removal has little measurable impact.
* Bigrams provide only a small performance gain relative to their increase in feature count.
* Increasing `min_df` improves efficiency and may slightly improve generalization.
* Feature extraction is currently the dominant computational cost in the pipeline.

These findings establish a strong baseline for future experiments involving alternative vectorization techniques and classification models.


## Model 2

Model - Logistic Regression     

Vectorization - CountVectorizer 

Features -      

* CountVectorizer  
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time :  44.2s 
* Training Time :  4.3s          

Metrics -           
* Accuracy:  0.97689
* Precision:  0.97102
* Recall:  0.98412
* F1:  0.97688
* Confusion Matrix:         
    [11292   357       -> False Positives            
     193     11963]     -> False Negatives              

In [23]:
# Creating features
start_time = time.perf_counter()

df = preprocess(df, check=True)
X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df, method="count")

end_time = time.perf_counter()
creation_time = end_time - start_time

print(f"\nFeatures created in {creation_time:.4f} seconds.")
print("Total features = ",total_features)

No of empty title rows:  563
No of empty text rows:  786
No of rows with empty title AND text:  1

Features created in 41.5606 seconds.
Total features =  477076


In [24]:
# Training model
start_time = time.perf_counter()

model = LogisticRegression(random_state=42).fit(X_train, Y_train)
Y_pred = model.predict(X_test)

end_time = time.perf_counter()
training_time = end_time - start_time

print(f"Training completed in {training_time:.4f} seconds.")

Training completed in 4.0630 seconds.


In [25]:
# Evaluation
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = evaluate(Y_test, Y_pred, model, feature_names)

Accuracy:  0.9768956101659315
Precision:  0.9710227272727273
Recall:  0.9841230667982889
F1:  0.9768772385072506

Confusion Matrix:
 [[11292   357]
 [  193 11963]]

Top 5 Negative Features:
                   Feature  Coefficient
352973             reuters    -4.595180
1810             breitbart    -4.431182
15986                  000    -2.227315
454520  washington reuters    -1.882917
15915           york times    -1.635496

Top 5 Positive Features:
          Feature  Coefficient
15125       video     3.126134
446457        via     2.807345
1777     breaking     1.512943
205953  image via     1.400511
288025    october     1.178850


In [26]:
# Storing results
results = []

results.append({
    "Experiment": "Baseline",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Features": total_features,
    "Feature Time (s)": creation_time,
    "Training Time (s)": training_time
})

## Experiments
* Removing article source names
* Varying min_df

### Experiment 1 (Source phrases removal)

In [27]:
results.append(
    run_experiment(
        df,
        vectorizer="count",
        experiment_name="Removed source names",
        remove_sourcewords=True
    )
)


Features created in 44.8281 seconds.
Total features =  475888
Training completed in 4.5053 seconds.
Accuracy:  0.9671077504725898
Precision:  0.9643936300530829
Recall:  0.9714544257979598
F1:  0.9670871193888291

Confusion Matrix:
 [[11213   436]
 [  347 11809]]

Top 5 Negative Features:
                 Feature  Coefficient
15784                000    -2.482234
167629            follow    -1.873979
321424  president donald    -1.859160
11706               says    -1.738267
95829                com    -1.572561

Top 5 Positive Features:
          Feature  Coefficient
14943       video     3.579886
445271        via     3.068323
1755     breaking     1.930871
205465  image via     1.485895
287406    october     1.364860


### Experiment 2(varying min_df)

In [28]:
results.append(
    run_experiment(
        df,
        vectorizer="count",
        experiment_name="min_df=2",
        mdf=2
    )
)


Features created in 40.3775 seconds.
Total features =  550000
Training completed in 4.8342 seconds.
Accuracy:  0.9765595463137996
Precision:  0.9716167859466494
Recall:  0.9828068443566963
F1:  0.9765422178869341

Confusion Matrix:
 [[11300   349]
 [  209 11947]]

Top 5 Negative Features:
                   Feature  Coefficient
412950             reuters    -4.571596
3627             breitbart    -4.404724
50001                  000    -2.271877
523666  washington reuters    -1.914797
49681           york times    -1.627311

Top 5 Positive Features:
          Feature  Coefficient
45708       video     3.187474
515119        via     2.889209
3558     breaking     1.500245
255316  image via     1.460988
344255    october     1.199232


In [29]:
results.append(
    run_experiment(
        df,
        vectorizer="count",
        experiment_name="min_df=10",
        mdf=10
    )
)


Features created in 41.0581 seconds.
Total features =  195987
Training completed in 3.1516 seconds.
Accuracy:  0.9759294265910523
Precision:  0.9702801461632156
Recall:  0.9829713721618953
F1:  0.9759105710676896

Confusion Matrix:
 [[11283   366]
 [  207 11949]]

Top 5 Negative Features:
                   Feature  Coefficient
899              breitbart    -4.670376
143912             reuters    -4.562232
7873                   000    -2.346193
186762  washington reuters    -1.952121
7832            york times    -1.741652

Top 5 Positive Features:
          Feature  Coefficient
7453        video     3.144598
183475        via     2.834549
886      breaking     1.638738
84276   image via     1.416352
117977    october     1.159267


In [30]:
results.append(
    run_experiment(
        df,
        vectorizer="count",
        experiment_name="min_df=20",
        mdf=20
    )
)


Features created in 39.6880 seconds.
Total features =  87429
Training completed in 2.6567 seconds.
Accuracy:  0.9752572988867885
Precision:  0.969554274579849
Recall:  0.9823955248436986
F1:  0.9752378001814137

Confusion Matrix:
 [[11274   375]
 [  214 11942]]

Top 5 Negative Features:
                  Feature  Coefficient
478             breitbart    -4.799800
64009             reuters    -4.493226
4191                  000    -2.511778
83498  washington reuters    -2.062168
4170           york times    -1.813128

Top 5 Positive Features:
         Feature  Coefficient
3962       video     3.277061
82076        via     2.897201
472     breaking     1.744742
37872  image via     1.486212
52735    october     1.169951


In [31]:
comparison_df = pd.DataFrame(results)
comparison_df.sort_values("Accuracy", ascending=False)

,Experiment,Accuracy,Precision,Recall,F1,Features,Feature Time (s),Training Time (s)
0,Baseline,0.976896,0.971023,0.984123,0.976877,477076,41.560594,4.063040
2,min_df=2,0.976560,0.971617,0.982807,0.976542,550000,40.377471,4.834223
3,min_df=10,0.975929,0.970280,0.982971,0.975911,195987,41.058135,3.151588
4,min_df=20,0.975257,0.969554,0.982396,0.975238,87429,39.688002,2.656665
1,Removed source names,0.967108,0.964394,0.971454,0.967087,475888,44.828117,4.505306


### Model 3
TF-IDF + Multinomial Naive Bayes

### Model 4
CountVectorizer + Naive Bayes

### Model 5
TF-IDF + Linear SVM

### Word Embeddings